# Projet NeRF-GPU biomédical — Stage ARSII
## Reconstruction 3D par NeRF-CT (variante Beer-Lambert, cone-beam)

Notebook autonome et nettoyé : crée l'arborescence, écrit tous les fichiers
sources, prépare les données CT réelles avec un simulateur cone-beam
cohérent avec la géométrie d'entraînement, entraîne le modèle, puis génère
les livrables visuels — y compris la **scène 3D tournante**.

**Changement clé vs les versions précédentes** : le simulateur de
projections n'utilise plus TIGRE (installation cassée sur Colab) ni une
approximation parallèle-beam (source du flou/bruit observé). Il utilise
désormais un simulateur cone-beam en PyTorch pur qui partage EXACTEMENT la
même géométrie de rayons que l'entraînement — cohérence garantie, et 100%
GPU/parallèle.

**Exécutez les cellules dans l'ordre.**

## 1. Arborescence du projet

In [ ]:
import os, sys
os.makedirs('nerf_parallele/src', exist_ok=True)
os.makedirs('nerf_parallele/scripts', exist_ok=True)
os.chdir('/content/nerf_parallele')
sys.path.insert(0, '/content/nerf_parallele/src')
print("Prêt. Répertoire courant :", os.getcwd())

## 2. Runtime GPU

In [ ]:
import torch
if torch.cuda.is_available():
    print(f"GPU disponible : {torch.cuda.get_device_name(0)}")
else:
    print("⚠ GPU non disponible — Exécution → Modifier le type d'exécution → T4 GPU, puis relancez.")

## 3. Dépendances

In [ ]:
!pip install scipy scikit-image imageio pydicom -q

## 4. Fichiers sources — projet semestriel (inchangés)

Ces trois fichiers ne sont pas modifiés par rapport au projet LEGO original.

In [ ]:
%%writefile src/nerf_model.py
# ════════════════════════════════
# Fichier : nerf_model.py
# Rôle    : Modèle NeRF (encodage, réseau, rayons, échantillonnage, rendu volumique)
# ════════════════════════════════

from __future__ import annotations

import math
from typing import Tuple

import torch
import torch.nn as nn
import torch.nn.functional as fonctionnel


class PositionalEncoding(nn.Module):
    """
    Encodage positionnel fréquentiel (NeRF) : pour chaque coordonnée, empile
    sin(2^k π x) et cos(2^k π x) pour k = 0 … L-1, puis concatène l’entrée brute.

    - Pour les positions 3D avec ``nb_frequences=10`` : sortie [N, 63] (3 + 6×10).
    - Pour les directions 3D avec ``nb_frequences=4`` : sortie [N, 27] (3 + 6×4).

    ⚡ PARALLÈLE : les opérations tensorielles s’exécutent en lot sur le GPU
    (vectorisation SIMD / cœurs CUDA) lorsque les tenseurs sont sur ``cuda``.
    """

    def __init__(self, nb_frequences: int = 10) -> None:
        super().__init__()
        self.nb_frequences = nb_frequences

    def forward(self, vecteurs: torch.Tensor) -> torch.Tensor:
        """
        Args:
            vecteurs: [N, 3] positions ou directions unitaires.

        Returns:
            Encodage [N, 3 + 6 * nb_frequences].
        """
        # ⚡ PARALLÈLE : construction du lot de features sur le device du tenseur d’entrée
        echelles = 2.0 ** torch.arange(
            self.nb_frequences,
            device=vecteurs.device,
            dtype=vecteurs.dtype,
        )
        parties = [vecteurs]
        for echelle in echelles:
            x = echelle * math.pi * vecteurs
            parties.append(torch.sin(x))
            parties.append(torch.cos(x))
        return torch.cat(parties, dim=-1)


class ReseauNeRF(nn.Module):
    """
    MLP NeRF : prédit la densité et la couleur dépendante de la vue.

    Architecture : 8 couches entièrement connectées avec connexion résiduelle
    (skip) depuis l’encodage de position à la 5ᵉ couche, puis tête RGB conditionnée
    par la direction encodée.

    ⚡ PARALLÈLE : un forward traite N points/rays en parallèle sur le GPU
    (multiplications matricielles batchées).
    """

    def __init__(
        self,
        dim_position: int = 63,
        dim_direction: int = 27,
        dim_cache: int = 256,
    ) -> None:
        super().__init__()
        self.dim_position = dim_position
        self.dim_direction = dim_direction
        self.dim_cache = dim_cache

        self.couches_position = nn.ModuleList()
        self.couches_position.append(nn.Linear(dim_position, dim_cache))
        for _ in range(3):
            self.couches_position.append(nn.Linear(dim_cache, dim_cache))
        self.couches_position.append(nn.Linear(dim_cache + dim_position, dim_cache))
        for _ in range(3):
            self.couches_position.append(nn.Linear(dim_cache, dim_cache))

        self.tete_densite = nn.Linear(dim_cache, 1)
        self.couche_feature = nn.Linear(dim_cache, dim_cache)
        self.tete_couleur = nn.Sequential(
            nn.Linear(dim_cache + dim_direction, dim_cache // 2),
            nn.ReLU(inplace=True),
            nn.Linear(dim_cache // 2, 3),
            nn.Sigmoid(),
        )

    def forward(
        self,
        position_encodee: torch.Tensor,
        direction_encodee: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            position_encodee: [N, dim_position]
            direction_encodee: [N, dim_direction]

        Returns:
            couleurs [N, 3] dans [0, 1], densites [N] (activées par ReLU dans le rendu volumique).
        """
        # ⚡ PARALLÈLE : propagation avant sur tout le batch de points
        x = position_encodee
        for index, couche in enumerate(self.couches_position):
            if index == 4:
                x = torch.cat([x, position_encodee], dim=-1)
            x = fonctionnel.relu(couche(x))

        densites = self.tete_densite(x).squeeze(-1)

        caracteristiques = self.couche_feature(x)
        entree_couleur = torch.cat([caracteristiques, direction_encodee], dim=-1)
        couleurs = self.tete_couleur(entree_couleur)
        return couleurs, densites


def generer_rayons(
    hauteur: int,
    largeur: int,
    focale: float,
    pose: torch.Tensor,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Génère les rayons pour tous les pixels : origines et directions unitaires monde.

    Convention type dataset synthétique NeRF : axe caméra -Z, grille centrée.

    Args:
        hauteur, largeur: résolution.
        focale: distance focale en pixels.
        pose: matrice 4×4 c2w (camera-to-world), tenseur [4, 4].

    Returns:
        origines [H*W, 3], directions [H*W, 3] normalisées (monde).
    """
    device = pose.device
    dtype = pose.dtype
    grille_i, grille_j = torch.meshgrid(
        torch.arange(largeur, device=device, dtype=dtype),
        torch.arange(hauteur, device=device, dtype=dtype),
        indexing="xy",
    )
    directions_camera = torch.stack(
        [
            (grille_i - largeur * 0.5 + 0.5) / focale,
            -(grille_j - hauteur * 0.5 + 0.5) / focale,
            -torch.ones_like(grille_i),
        ],
        dim=-1,
    )
    directions_camera = directions_camera / directions_camera.norm(dim=-1, keepdim=True)

    rotation = pose[:3, :3]
    translation = pose[:3, 3]
    directions_monde = (rotation @ directions_camera.reshape(-1, 3).T).T
    directions_monde = directions_monde / (directions_monde.norm(dim=-1, keepdim=True) + 1e-8)
    origines = translation.expand(hauteur * largeur, 3)
    return origines, directions_monde


def echantillonner_points(
    origines: torch.Tensor,
    directions: torch.Tensor,
    proche: float,
    loin: float,
    nb_points: int,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Échantillonne des points le long de chaque rayon entre ``proche`` et ``loin``.

    Args:
        origines: [N_rayons, 3]
        directions: [N_rayons, 3] unitaires
        proche, loin: bornes le long du rayon (t)
        nb_points: nombre d’échantillons par rayon

    Returns:
        points: [N_rayons, nb_points, 3]
        t_vals: [N_rayons, nb_points] distances le long du rayon
    """
    # ⚡ PARALLÈLE : toutes les combinaisons rayons × échantillons calculées en tenseur
    t_vals = torch.linspace(
        proche,
        loin,
        nb_points,
        device=origines.device,
        dtype=origines.dtype,
    )
    t_vals = t_vals.expand(origines.shape[0], nb_points)
    directions_etendues = directions.unsqueeze(1).expand(-1, nb_points, -1)
    t_etendu = t_vals.unsqueeze(-1)
    points = origines.unsqueeze(1) + t_etendu * directions_etendues
    return points, t_vals


def volume_rendering(
    couleurs: torch.Tensor,
    densites: torch.Tensor,
    t_vals: torch.Tensor,
) -> torch.Tensor:
    """
    Rendu volumique par accumulation alpha (NeRF).

    Args:
        couleurs: [N_rayons, nb_points, 3]
        densites: [N_rayons, nb_points] valeurs σ ≥ 0
        t_vals: [N_rayons, nb_points] paramètres t le long du rayon

    Returns:
        couleur_finale [N_rayons, 3]
    """
    # ⚡ PARALLÈLE : somme pondérée sur la dimension des échantillons pour tous les rayons
    deltas = t_vals[..., 1:] - t_vals[..., :-1]
    delta_fin = 1e10 * torch.ones_like(deltas[..., :1])
    deltas = torch.cat([deltas, delta_fin], dim=-1)

    alpha = 1.0 - torch.exp(-fonctionnel.relu(densites) * deltas)
    transmittance = torch.cumprod(
        torch.cat(
            [
                torch.ones_like(alpha[..., :1]),
                1.0 - alpha[..., :-1] + 1e-10,
            ],
            dim=-1,
        ),
        dim=-1,
    )
    poids = alpha * transmittance
    couleur_finale = (poids.unsqueeze(-1) * couleurs).sum(dim=-2)
    return couleur_finale


In [ ]:
%%writefile src/dataset.py
# ════════════════════════════════
# Fichier : dataset.py
# Rôle    : Chargement du dataset synthétique NeRF et préparation des rayons
# ════════════════════════════════

from __future__ import annotations

import json
import sys
from pathlib import Path

_REPERTOIRE_SRC = Path(__file__).resolve().parent
if str(_REPERTOIRE_SRC) not in sys.path:
    sys.path.insert(0, str(_REPERTOIRE_SRC))
from typing import Tuple

import numpy as np
import torch
from PIL import Image

from nerf_model import generer_rayons


def charger_dataset(
    dossier: str | Path,
    split: str,
    taille: Tuple[int, int] = (100, 100),
) -> Tuple[np.ndarray, np.ndarray, float]:
    """
    Lit ``transforms_{split}.json`` (format Blender / NeRF synthétique) et charge
    les images PNG en RGBA, valeurs flottantes dans ``[0, 1]``.

    Args:
        dossier: répertoire racine du scène (ex. ``.../lego``).
        split: ``"train"`` ou ``"val"`` / ``"test"`` selon le fichier disponible.
        taille: ``(H, W)`` après redimensionnement.

    Returns:
        images: ``[N, H, W, 4]`` float32
        poses: ``[N, 4, 4]`` float32 (camera-to-world)
        focale: distance focale en **pixels**, cohérente avec ``taille[1]`` (largeur).
    """
    dossier = Path(dossier)
    chemin_json = dossier / f"transforms_{split}.json"
    if not chemin_json.is_file():
        raise FileNotFoundError(f"Fichier introuvable : {chemin_json}")

    with open(chemin_json, encoding="utf-8") as fichier:
        meta = json.load(fichier)

    angle_camera_x = float(meta["camera_angle_x"])
    cadres = meta["frames"]
    hauteur, largeur = taille

    # Focale en pixels pour la largeur cible (même convention que le rendu NeRF).
    focale = 0.5 * largeur / np.tan(0.5 * angle_camera_x)

    images_liste: list[np.ndarray] = []
    poses_liste: list[np.ndarray] = []

    for cadre in cadres:
        chemin_relatif = cadre["file_path"]
        if chemin_relatif.startswith("./"):
            chemin_relatif = chemin_relatif[2:]
        base = Path(chemin_relatif)
        candidats = [
            dossier / f"{chemin_relatif}.png",
            dossier / base.with_suffix(".png"),
            dossier / chemin_relatif,
        ]
        chemin_image = next((c for c in candidats if c.is_file()), None)
        if chemin_image is None:
            raise FileNotFoundError(
                f"Image introuvable pour {cadre['file_path']} (essayé : {candidats[0]})"
            )

        avec_image = Image.open(chemin_image).convert("RGBA")
        avec_image = avec_image.resize((largeur, hauteur), Image.BICUBIC)
        tableau = np.asarray(avec_image).astype(np.float32) / 255.0
        images_liste.append(tableau)

        matrice = np.array(cadre["transform_matrix"], dtype=np.float32).reshape(4, 4)
        poses_liste.append(matrice)

    images = np.stack(images_liste, axis=0)
    poses = np.stack(poses_liste, axis=0)
    return images, poses, float(focale)


def preparer_rayons(
    images: np.ndarray,
    poses: np.ndarray,
    focale: float,
    device: torch.device | str,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Précalcule **tous** les rayons du jeu de données : une ligne par pixel de chaque image.

    Args:
        images: ``[N, H, W, 4]`` (numpy ou compatible).
        poses: ``[N, 4, 4]``.
        focale: focale en pixels.
        device: ``torch.device`` cible.

    Returns:
        all_rayons_o, all_rayons_d, all_couleurs_cibles — chacun ``[N * H * W, 3]``.
    """
    device = torch.device(device)
    if isinstance(images, torch.Tensor):
        images_np = images.detach().cpu().numpy()
    else:
        images_np = np.asarray(images)

    poses_t = torch.as_tensor(poses, dtype=torch.float32, device=device)
    nombre_images, hauteur, largeur, _ = images_np.shape

    # ⚡ PARALLÈLE : chaque ``generer_rayons`` produit des tenseurs GPU ; la boucle reste séquentielle par image.
    blocs_origines: list[torch.Tensor] = []
    blocs_directions: list[torch.Tensor] = []
    blocs_couleurs: list[torch.Tensor] = []

    for indice in range(nombre_images):
        pose = poses_t[indice]
        origines, directions = generer_rayons(
            hauteur,
            largeur,
            float(focale),
            pose,
        )
        couleurs = torch.as_tensor(
            images_np[indice, ..., :3],
            dtype=torch.float32,
            device=device,
        ).reshape(-1, 3)

        blocs_origines.append(origines)
        blocs_directions.append(directions)
        blocs_couleurs.append(couleurs)

    return (
        torch.cat(blocs_origines, dim=0),
        torch.cat(blocs_directions, dim=0),
        torch.cat(blocs_couleurs, dim=0),
    )


In [ ]:
%%writefile src/train.py
# ════════════════════════════════
# Fichier : train.py
# Rôle    : Boucle d’entraînement NeRF (mixed precision, journalisation, checkpoints)
# ════════════════════════════════

from __future__ import annotations

import sys
import time
from pathlib import Path

_REPERTOIRE_SRC = Path(__file__).resolve().parent
if str(_REPERTOIRE_SRC) not in sys.path:
    sys.path.insert(0, str(_REPERTOIRE_SRC))

from typing import Any, Dict, List, Tuple

import torch
import torch.nn as nn
from contextlib import nullcontext
from torch import amp

from dataset import preparer_rayons
from nerf_model import PositionalEncoding, ReseauNeRF, echantillonner_points, volume_rendering


def _psnr_batch(prediction: torch.Tensor, cible: torch.Tensor) -> float:
    mse = torch.mean((prediction - cible) ** 2).clamp(min=1e-10)
    return float((-10.0 * torch.log10(mse)).item())


def entrainer_nerf(
    images_train: Any,
    poses_train: Any,
    focale: float,
    nb_iterations: int = 5000,
    taille_batch: int = 1024,
    nb_points: int = 64,
    lr: float = 5e-4,
    proche: float = 2.0,
    loin: float = 6.0,
    sauvegarder_tous: int = 2500,
    dossier_checkpoints: str = "checkpoints/",
) -> Tuple[nn.Module, Dict[str, Any]]:
    """
    Entraîne un ``ReseauNeRF`` avec précision mixte FP16 sur GPU si disponible.

    Les encodeurs positionnels n’ont pas de poids apprenables ; seul le MLP est optimisé.

    Returns:
        modele entraîné, dictionnaire ``historique`` avec
        ``losses``, ``psnrs``, ``temps_par_iter``, ``enc_position``, ``enc_direction``.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    enc_position = PositionalEncoding(nb_frequences=10).to(device)
    enc_direction = PositionalEncoding(nb_frequences=4).to(device)
    modele = ReseauNeRF(
        dim_position=63,
        dim_direction=27,
        dim_cache=256,
    ).to(device)

    optimiseur = torch.optim.Adam(modele.parameters(), lr=lr)
    echelleur = amp.GradScaler("cuda", enabled=(device.type == "cuda"))

    all_rayons_o, all_rayons_d, all_couleurs = preparer_rayons(
        images_train, poses_train, focale, device
    )
    nombre_rayons = all_rayons_o.shape[0]

    dossier_ckpt = Path(dossier_checkpoints)
    dossier_ckpt.mkdir(parents=True, exist_ok=True)

    losses: List[float] = []
    psnrs: List[float] = []
    temps_par_iter: List[float] = []

    for iteration in range(nb_iterations):
        t_debut = time.perf_counter()

        indices = torch.randint(
            0,
            nombre_rayons,
            (taille_batch,),
            device=device,
        )
        origines = all_rayons_o[indices]
        directions = all_rayons_d[indices]
        couleurs_cibles = all_couleurs[indices]

        optimiseur.zero_grad(set_to_none=True)

        # ⚡ PARALLÈLE : forward + backward FP16 sur GPU (Tensor Cores)
        contexte_precision = (
            amp.autocast("cuda") if device.type == "cuda" else nullcontext()
        )
        with contexte_precision:
            points, t_vals = echantillonner_points(
                origines, directions, proche, loin, nb_points
            )
            points_plats = points.reshape(-1, 3)
            directions_rep = directions.unsqueeze(1).expand(-1, nb_points, -1).reshape(-1, 3)

            directions_unitaires = directions_rep / (
                directions_rep.norm(dim=-1, keepdim=True) + 1e-8
            )
            position_encodee = enc_position(points_plats)
            direction_encodee = enc_direction(directions_unitaires)

            couleurs, densites = modele(position_encodee, direction_encodee)
            couleurs = couleurs.view(taille_batch, nb_points, 3)
            densites = densites.view(taille_batch, nb_points)

            couleurs_rendues = volume_rendering(couleurs, densites, t_vals)
            perte = torch.mean((couleurs_rendues - couleurs_cibles) ** 2)

        if device.type == "cuda":
            echelleur.scale(perte).backward()
            echelleur.step(optimiseur)
            echelleur.update()
        else:
            perte.backward()
            optimiseur.step()

        t_fin = time.perf_counter()
        temps_par_iter.append((t_fin - t_debut) * 1000.0)
        losses.append(float(perte.item()))

        with torch.no_grad():
            psnr_batch = _psnr_batch(couleurs_rendues.detach(), couleurs_cibles)
        psnrs.append(psnr_batch)

        if (iteration + 1) % 500 == 0:
            print(
                f"[{iteration + 1}/{nb_iterations}] "
                f"loss={perte.item():.6f}  PSNR≈{psnr_batch:.2f} dB"
            )

        if sauvegarder_tous > 0 and (iteration + 1) % sauvegarder_tous == 0:
            chemin = dossier_ckpt / f"nerf_iter_{iteration + 1:06d}.pt"
            torch.save(
                {
                    "modele": modele.state_dict(),
                    "enc_position": enc_position.state_dict(),
                    "enc_direction": enc_direction.state_dict(),
                    "optimiseur": optimiseur.state_dict(),
                    "iteration": iteration + 1,
                },
                chemin,
            )
            print(f"Checkpoint sauvegardé : {chemin}")

    historique = {
        "losses": losses,
        "psnrs": psnrs,
        "temps_par_iter": temps_par_iter,
        "enc_position": enc_position,
        "enc_direction": enc_direction,
    }
    return modele, historique


## 5. Fichiers sources — variante CT biomédicale

- `geometrie_ct.py` : géométrie cone-beam + rayons + **simulateur de
  projections cohérent** (remplace TIGRE et l'approximation parallèle-beam)
- `nerf_model_ct.py` : tête d'atténuation + rendu Beer-Lambert
- `dataset_ct.py` : chargement des projections + préparation des rayons
- `train_ct.py` : boucle d'entraînement (mixed precision, LR decay, gradient clipping)
- `visualiser_ct.py` : retours visuels 2D (vérité vs rendu, nouvelles vues, coupe)
- `scene3d_ct.py` : **scène 3D tournante** (mesh + rotation 360°, GIF)

In [ ]:
%%writefile src/geometrie_ct.py
# ════════════════════════════════
# Fichier : geometrie_ct.py
# Rôle    : Géométrie cone-beam CT + génération des rayons source → détecteur
#           + simulateur de projections cone-beam en PyTorch pur.
#
# CHOIX TECHNIQUE IMPORTANT : le simulateur de projections
# (simuler_projections_conebeam) utilise EXACTEMENT la même fonction de rayons
# (generer_rayons_ct) que l'entraînement (train_ct.py). C'est volontaire et
# critique : ça garantit que la géométrie utilisée pour générer les données
# d'entraînement et celle utilisée pour les apprendre sont identiques.
# Un décalage entre les deux (ex. simulateur parallèle-beam vs modèle
# cone-beam) produit exactement le type de flou/incohérence observé avec
# l'approche précédente. On n'a plus besoin de TIGRE : tout tourne en
# PyTorch, donc sur GPU, en parallèle sur tous les pixels/rayons.
# ════════════════════════════════

from __future__ import annotations

from dataclasses import dataclass
from typing import Tuple

import numpy as np
import torch
import torch.nn.functional as F

from nerf_model import echantillonner_points  # réutilisé tel quel, pas de duplication


@dataclass
class GeometrieCT:
    """
    Paramètres géométriques d'un scan cone-beam (unités : mm).

    DSO: distance source → centre du volume (isocentre).
    DSD: distance source → détecteur.
    n_detecteur: (lignes, colonnes) du détecteur en pixels.
    taille_pixel_detecteur: (hauteur, largeur) d'un pixel détecteur en mm.
    taille_volume: (Z, Y, X) taille physique du volume en mm — le volume est
        supposé centré à l'origine, axe Z = axe de rotation (axe patient).
    """

    DSO: float
    DSD: float
    n_detecteur: Tuple[int, int]
    taille_pixel_detecteur: Tuple[float, float]
    taille_volume: Tuple[float, float, float]


def generer_rayons_ct(
    geometrie: GeometrieCT,
    angle_rad: float,
    device: torch.device | str = "cpu",
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Génère les rayons source → pixel détecteur pour UN angle de scan.
    Convention : rotation autour de l'axe Z (axe patient), volume centré à
    l'origine. Composantes des points/rayons dans l'ordre (x, y, z).

    Returns:
        origines: [n_pixels, 3], directions: [n_pixels, 3] (unitaires).
    """
    device = torch.device(device)
    n_lignes, n_colonnes = geometrie.n_detecteur
    ph, pw = geometrie.taille_pixel_detecteur

    u = (torch.arange(n_colonnes, device=device, dtype=torch.float32) - (n_colonnes - 1) / 2.0) * pw
    v = (torch.arange(n_lignes, device=device, dtype=torch.float32) - (n_lignes - 1) / 2.0) * ph
    grille_v, grille_u = torch.meshgrid(v, u, indexing="ij")

    points_detecteur_local = torch.stack(
        [
            torch.full_like(grille_u, geometrie.DSD - geometrie.DSO),
            grille_u,
            grille_v,
        ],
        dim=-1,
    ).reshape(-1, 3)

    source_local = torch.tensor([-geometrie.DSO, 0.0, 0.0], device=device, dtype=torch.float32)

    cos_a, sin_a = float(np.cos(angle_rad)), float(np.sin(angle_rad))
    rotation = torch.tensor(
        [[cos_a, -sin_a, 0.0], [sin_a, cos_a, 0.0], [0.0, 0.0, 1.0]],
        device=device,
        dtype=torch.float32,
    )

    source_monde = rotation @ source_local
    points_detecteur_monde = (rotation @ points_detecteur_local.T).T

    directions = points_detecteur_monde - source_monde.unsqueeze(0)
    directions = directions / (directions.norm(dim=-1, keepdim=True) + 1e-8)
    origines = source_monde.unsqueeze(0).expand(directions.shape[0], 3)

    return origines, directions


def bornes_integration(geometrie: GeometrieCT) -> Tuple[float, float]:
    """Bornes (proche, loin) le long du rayon, dérivées de la géométrie."""
    demi_diag = 0.5 * float(np.linalg.norm(geometrie.taille_volume))
    proche = max(geometrie.DSO - demi_diag, 0.0)
    loin = geometrie.DSO + demi_diag
    return proche, loin


@torch.no_grad()
def simuler_projections_conebeam(
    volume: np.ndarray,
    geometrie: GeometrieCT,
    angles: np.ndarray,
    device: torch.device | str = "cpu",
    nb_points_integration: int = 256,
    taille_lot_pixels: int = 16384,
) -> np.ndarray:
    """
    Simule des projections cone-beam d'un volume connu, EN UTILISANT LA MÊME
    géométrie de rayons que l'entraînement (generer_rayons_ct). C'est le
    remplacement direct de TIGRE / de l'approximation parallèle-beam.

    Le volume est échantillonné par interpolation trilinéaire (grid_sample)
    le long de chaque rayon, puis intégré (loi de Beer-Lambert : la cible
    d'entraînement est l'intégrale de ligne ∫μ dt, pas l'intensité transmise).

    Args:
        volume: [Z, Y, X] coefficients d'atténuation (float32).
        angles: angles de scan en radians.
        nb_points_integration: nombre de points le long du rayon pour LA
            SIMULATION (peut/doit être plus élevé que nb_points utilisé à
            l'entraînement du NeRF, pour que la "vérité terrain" soit plus
            précise que ce que le modèle a besoin d'apprendre).

    Returns:
        projections [n_angles, n_lignes, n_colonnes] (float32).
    """
    device = torch.device(device)
    volume_t = torch.as_tensor(volume, dtype=torch.float32, device=device)
    volume_5d = volume_t.unsqueeze(0).unsqueeze(0)  # [1, 1, Z, Y, X]

    taille_z, taille_y, taille_x = geometrie.taille_volume
    demi_taille = torch.tensor([taille_x / 2, taille_y / 2, taille_z / 2], device=device)

    proche, loin = bornes_integration(geometrie)

    projections = []
    for angle in angles:
        origines, directions = generer_rayons_ct(geometrie, float(angle), device=device)
        n_pixels = origines.shape[0]
        integrale_totale = torch.zeros(n_pixels, device=device)

        for debut in range(0, n_pixels, taille_lot_pixels):
            o = origines[debut : debut + taille_lot_pixels]
            d = directions[debut : debut + taille_lot_pixels]
            points, t_vals = echantillonner_points(o, d, proche, loin, nb_points_integration)

            # Normalisation pour grid_sample : coordonnées (x,y,z) dans
            # [-1, 1], volume centré à l'origine (même convention que
            # bornes_integration / generer_rayons_ct).
            grille = (points / demi_taille).clamp(-1.0, 1.0)
            n_local = o.shape[0]
            grille = grille.reshape(1, 1, 1, n_local * nb_points_integration, 3)

            valeurs = F.grid_sample(
                volume_5d, grille, mode="bilinear", align_corners=True, padding_mode="zeros"
            )
            valeurs = valeurs.reshape(n_local, nb_points_integration)

            deltas = t_vals[..., 1:] - t_vals[..., :-1]
            deltas = torch.cat([deltas, deltas[..., -1:]], dim=-1)
            integrale_totale[debut : debut + taille_lot_pixels] = (valeurs * deltas).sum(dim=-1)

        projections.append(integrale_totale.reshape(geometrie.n_detecteur).cpu())

    return torch.stack(projections).numpy().astype(np.float32)


In [ ]:
%%writefile src/nerf_model_ct.py
# ════════════════════════════════
# Fichier : nerf_model_ct.py
# Rôle    : Variante CT du modèle NeRF — prédit un coefficient d'atténuation μ
#           (1 valeur, indépendante de la direction de vue) et rend par
#           intégrale de ligne (Beer-Lambert).
#
# RÉUTILISE tel quel depuis nerf_model.py : PositionalEncoding.
# ════════════════════════════════

from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as fonctionnel

from nerf_model import PositionalEncoding  # noqa: F401 (réexporté pour commodité)


class ReseauNeRF_CT(nn.Module):
    """
    MLP prédisant un coefficient d'atténuation μ(x) ≥ 0 pour chaque point 3D.
    Pas d'encodage de direction : μ est une propriété physique du point,
    indépendante de l'angle d'observation.
    """

    def __init__(self, dim_position: int = 63, dim_cache: int = 256) -> None:
        super().__init__()
        self.couches_position = nn.ModuleList()
        self.couches_position.append(nn.Linear(dim_position, dim_cache))
        for _ in range(3):
            self.couches_position.append(nn.Linear(dim_cache, dim_cache))
        self.couches_position.append(nn.Linear(dim_cache + dim_position, dim_cache))
        for _ in range(3):
            self.couches_position.append(nn.Linear(dim_cache, dim_cache))

        self.tete_attenuation = nn.Linear(dim_cache, 1)

    def forward(self, position_encodee: torch.Tensor) -> torch.Tensor:
        x = position_encodee
        for index, couche in enumerate(self.couches_position):
            if index == 4:
                x = torch.cat([x, position_encodee], dim=-1)
            x = fonctionnel.relu(couche(x))
        return fonctionnel.softplus(self.tete_attenuation(x).squeeze(-1))


def volume_rendering_beerlambert(attenuation: torch.Tensor, t_vals: torch.Tensor) -> torch.Tensor:
    """
    Rendu par intégrale de ligne (Beer-Lambert) : ∫ μ dt le long du rayon,
    par somme de Riemann. Utilisé à la fois par l'entraînement (train_ct.py)
    et implicitement cohérent avec le simulateur de données
    (geometrie_ct.simuler_projections_conebeam), qui utilise la même formule.
    """
    deltas = t_vals[..., 1:] - t_vals[..., :-1]
    deltas = torch.cat([deltas, deltas[..., -1:]], dim=-1)
    return (attenuation * deltas).sum(dim=-1)


In [ ]:
%%writefile src/dataset_ct.py
# ════════════════════════════════
# Fichier : dataset_ct.py
# Rôle    : Charge les projections CT simulées + leur géométrie, et prépare
#           les rayons pour l'entraînement.
# ════════════════════════════════

from __future__ import annotations

import json
import os
from typing import Dict, Tuple

import numpy as np
import torch

from geometrie_ct import GeometrieCT, generer_rayons_ct


def charger_dataset_ct(dossier: str, split: str = "train") -> Dict:
    """
    Structure attendue :
        dossier/
          geometrie.json
          {split}/angles.npy, projections.npy
    """
    with open(os.path.join(dossier, "geometrie.json"), "r") as f:
        params_geo = json.load(f)
    geometrie = GeometrieCT(
        DSO=params_geo["DSO"],
        DSD=params_geo["DSD"],
        n_detecteur=tuple(params_geo["n_detecteur"]),
        taille_pixel_detecteur=tuple(params_geo["taille_pixel_detecteur"]),
        taille_volume=tuple(params_geo["taille_volume"]),
    )

    sous_dossier = os.path.join(dossier, split)
    angles = np.load(os.path.join(sous_dossier, "angles.npy"))
    projections = np.load(os.path.join(sous_dossier, "projections.npy"))

    return {
        "projections": torch.from_numpy(projections).float(),
        "angles": torch.from_numpy(angles).float(),
        "geometrie": geometrie,
    }


def preparer_rayons_ct(donnees: Dict, device: torch.device | str = "cpu") -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Pré-calcule tous les rayons (un par pixel détecteur, par angle) + cibles."""
    device = torch.device(device)
    geometrie: GeometrieCT = donnees["geometrie"]
    angles = donnees["angles"]
    projections = donnees["projections"]

    toutes_origines, toutes_directions, toutes_cibles = [], [], []
    for i, angle in enumerate(angles.tolist()):
        origines, directions = generer_rayons_ct(geometrie, angle, device=device)
        toutes_origines.append(origines)
        toutes_directions.append(directions)
        toutes_cibles.append(projections[i].reshape(-1).to(device))

    return (
        torch.cat(toutes_origines, dim=0),
        torch.cat(toutes_directions, dim=0),
        torch.cat(toutes_cibles, dim=0),
    )


In [ ]:
%%writefile src/train_ct.py
# ════════════════════════════════
# Fichier : train_ct.py
# Rôle    : Boucle d'entraînement NeRF-CT — mixed precision, checkpoints,
#           + décroissance du learning rate (nouveau — améliore la
#           convergence fine par rapport à un LR constant).
# ════════════════════════════════

from __future__ import annotations

import sys
import time
from contextlib import nullcontext
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import torch
import torch.nn as nn
from torch import amp

_REPERTOIRE_SRC = Path(__file__).resolve().parent
if str(_REPERTOIRE_SRC) not in sys.path:
    sys.path.insert(0, str(_REPERTOIRE_SRC))

from dataset_ct import charger_dataset_ct, preparer_rayons_ct
from geometrie_ct import bornes_integration
from nerf_model import PositionalEncoding
from nerf_model_ct import ReseauNeRF_CT, volume_rendering_beerlambert


def _psnr_batch(prediction: torch.Tensor, cible: torch.Tensor) -> float:
    mse = torch.mean((prediction - cible) ** 2).clamp(min=1e-10)
    return float((-10.0 * torch.log10(mse)).item())


def entrainer_nerf_ct(
    dossier_dataset: str,
    nb_iterations: int = 8000,
    taille_batch: int = 2048,
    nb_points: int = 128,
    lr: float = 5e-4,
    lr_decay_final: float = 0.1,
    sauvegarder_tous: int = 1000,
    dossier_checkpoints: str = "checkpoints_ct/",
    checkpoint_reprise: Optional[str] = None,
) -> Tuple[nn.Module, Dict[str, Any]]:
    """
    Entraîne un ReseauNeRF_CT avec rendu Beer-Lambert.

    Changements vs la version Sprint 1 :
      - nb_iterations/taille_batch/nb_points par défaut augmentés (qualité
        finale plutôt que juste "code tournable").
      - Décroissance exponentielle du LR (5e-4 → 5e-4*lr_decay_final) —
        stabilise la convergence en fin d'entraînement.
      - checkpoint_reprise : reprendre un entraînement interrompu.

    Returns:
        modele entraîné, historique (losses/psnrs/temps_par_iter/enc_position).
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True  # tailles d'entrée fixes → accélère les convs/matmuls répétées

    donnees = charger_dataset_ct(dossier_dataset, split="train")
    geometrie = donnees["geometrie"]
    proche, loin = bornes_integration(geometrie)
    print(f"Bornes d'intégration : proche={proche:.1f}mm loin={loin:.1f}mm")

    enc_position = PositionalEncoding(nb_frequences=10).to(device)
    modele = ReseauNeRF_CT(dim_position=63, dim_cache=256).to(device)

    optimiseur = torch.optim.Adam(modele.parameters(), lr=lr)
    # Décroissance exponentielle : lr(iter) = lr * lr_decay_final^(iter/nb_iterations)
    gamma = lr_decay_final ** (1.0 / max(nb_iterations, 1))
    ordonnanceur = torch.optim.lr_scheduler.ExponentialLR(optimiseur, gamma=gamma)
    echelleur = amp.GradScaler("cuda", enabled=(device.type == "cuda"))

    iteration_depart = 0
    if checkpoint_reprise is not None:
        ckpt = torch.load(checkpoint_reprise, map_location=device)
        modele.load_state_dict(ckpt["modele"])
        enc_position.load_state_dict(ckpt["enc_position"])
        optimiseur.load_state_dict(ckpt["optimiseur"])
        iteration_depart = ckpt["iteration"]
        print(f"Reprise depuis {checkpoint_reprise} (itération {iteration_depart})")

    all_rayons_o, all_rayons_d, all_cibles = preparer_rayons_ct(donnees, device)
    nombre_rayons = all_rayons_o.shape[0]
    print(f"Total rayons préparés : {nombre_rayons}")

    dossier_ckpt = Path(dossier_checkpoints)
    dossier_ckpt.mkdir(parents=True, exist_ok=True)

    losses: List[float] = []
    psnrs: List[float] = []
    temps_par_iter: List[float] = []

    t_debut_total = time.perf_counter()

    for iteration in range(iteration_depart, nb_iterations):
        t_debut = time.perf_counter()

        indices = torch.randint(0, nombre_rayons, (taille_batch,), device=device)
        origines = all_rayons_o[indices]
        directions = all_rayons_d[indices]
        cibles = all_cibles[indices]

        optimiseur.zero_grad(set_to_none=True)

        contexte_precision = amp.autocast("cuda") if device.type == "cuda" else nullcontext()
        with contexte_precision:
            points, t_vals = _echantillonner(origines, directions, proche, loin, nb_points)
            position_encodee = enc_position(points.reshape(-1, 3))
            attenuation = modele(position_encodee).view(taille_batch, nb_points)
            integrale_rendue = volume_rendering_beerlambert(attenuation, t_vals)
            perte = torch.mean((integrale_rendue - cibles) ** 2)

        if device.type == "cuda":
            echelleur.scale(perte).backward()
            echelleur.unscale_(optimiseur)
            torch.nn.utils.clip_grad_norm_(modele.parameters(), max_norm=1.0)
            echelleur.step(optimiseur)
            echelleur.update()
        else:
            perte.backward()
            torch.nn.utils.clip_grad_norm_(modele.parameters(), max_norm=1.0)
            optimiseur.step()
        ordonnanceur.step()

        temps_par_iter.append((time.perf_counter() - t_debut) * 1000.0)
        losses.append(float(perte.item()))
        with torch.no_grad():
            psnrs.append(_psnr_batch(integrale_rendue.detach(), cibles))

        if (iteration + 1) % 100 == 0:
            lr_actuel = ordonnanceur.get_last_lr()[0]
            ecoule = time.perf_counter() - t_debut_total
            print(
                f"[{iteration + 1}/{nb_iterations}] loss={perte.item():.4f} "
                f"PSNR≈{psnrs[-1]:.2f}dB lr={lr_actuel:.2e} temps={ecoule/60:.1f}min"
            )

        if sauvegarder_tous > 0 and (iteration + 1) % sauvegarder_tous == 0:
            chemin = dossier_ckpt / f"nerf_ct_iter_{iteration + 1:06d}.pt"
            torch.save(
                {
                    "modele": modele.state_dict(),
                    "enc_position": enc_position.state_dict(),
                    "optimiseur": optimiseur.state_dict(),
                    "iteration": iteration + 1,
                },
                chemin,
            )
            print(f"Checkpoint sauvegardé : {chemin}")

    historique = {
        "losses": losses,
        "psnrs": psnrs,
        "temps_par_iter": temps_par_iter,
        "enc_position": enc_position,
    }
    return modele, historique


def _echantillonner(origines, directions, proche, loin, nb_points):
    """Échantillonnage le long du rayon (identique à nerf_model.echantillonner_points)."""
    t_vals = torch.linspace(proche, loin, nb_points, device=origines.device, dtype=origines.dtype)
    t_vals = t_vals.expand(origines.shape[0], nb_points)
    directions_etendues = directions.unsqueeze(1).expand(-1, nb_points, -1)
    points = origines.unsqueeze(1) + t_vals.unsqueeze(-1) * directions_etendues
    return points, t_vals


if __name__ == "__main__":
    dossier = sys.argv[1] if len(sys.argv) > 1 else "data/ct_pancreas_pilote"
    entrainer_nerf_ct(dossier, nb_iterations=200, sauvegarder_tous=100)


In [ ]:
%%writefile src/visualiser_ct.py
# ════════════════════════════════
# Fichier : visualiser_ct.py
# Rôle    : Retours visuels 2D — projections rendues vs vérité, nouvelles
#           vues, coupe 2D reconstruite, courbes d'entraînement.
# Pour la scène 3D tournante (mesh + turntable), voir scene3d_ct.py.
# ════════════════════════════════

from __future__ import annotations

import os
from typing import List, Optional

import matplotlib.pyplot as plt
import numpy as np
import torch

from geometrie_ct import GeometrieCT, generer_rayons_ct, bornes_integration
from nerf_model import echantillonner_points
from nerf_model_ct import volume_rendering_beerlambert


@torch.no_grad()
def rendre_projection_ct(
    modele, enc_position, geometrie: GeometrieCT, angle_rad: float,
    proche: float, loin: float, nb_points: int = 128,
    device: torch.device | str = "cpu", taille_batch: int = 8192,
) -> np.ndarray:
    """Rend une projection complète pour un angle donné (vu ou non à l'entraînement)."""
    device = torch.device(device)
    origines, directions = generer_rayons_ct(geometrie, angle_rad, device=device)
    n_pixels = origines.shape[0]

    sorties = []
    for debut in range(0, n_pixels, taille_batch):
        o = origines[debut : debut + taille_batch]
        d = directions[debut : debut + taille_batch]
        points, t_vals = echantillonner_points(o, d, proche, loin, nb_points)
        position_encodee = enc_position(points.reshape(-1, 3))
        attenuation = modele(position_encodee).view(o.shape[0], nb_points)
        sorties.append(volume_rendering_beerlambert(attenuation, t_vals).cpu())

    return torch.cat(sorties).reshape(geometrie.n_detecteur).numpy()


@torch.no_grad()
def rendre_coupe_ct(
    modele, enc_position, geometrie: GeometrieCT, axe: str = "z",
    niveau: float = 0.0, resolution: int = 128, device: torch.device | str = "cpu",
) -> np.ndarray:
    """Coupe 2D du champ d'atténuation appris — plus parlant qu'une projection."""
    device = torch.device(device)
    demi = [t / 2 for t in geometrie.taille_volume]

    lin1 = torch.linspace(-demi[1], demi[1], resolution, device=device)
    lin2 = torch.linspace(-demi[2], demi[2], resolution, device=device)
    grille1, grille2 = torch.meshgrid(lin1, lin2, indexing="ij")

    if axe == "z":
        points = torch.stack([torch.full_like(grille1, niveau), grille1, grille2], dim=-1)
    elif axe == "y":
        points = torch.stack([grille1, torch.full_like(grille1, niveau), grille2], dim=-1)
    else:
        points = torch.stack([grille1, grille2, torch.full_like(grille1, niveau)], dim=-1)

    position_encodee = enc_position(points.reshape(-1, 3))
    attenuation = modele(position_encodee).reshape(resolution, resolution)
    return attenuation.cpu().numpy()


def afficher_grille_projections(images: List[np.ndarray], titres: List[str], titre_general: str = "", cmap: str = "gray") -> plt.Figure:
    n = len(images)
    n_colonnes = min(5, n)
    n_lignes = int(np.ceil(n / n_colonnes))
    fig, axes = plt.subplots(n_lignes, n_colonnes, figsize=(3 * n_colonnes, 3 * n_lignes))
    axes = np.atleast_1d(axes).reshape(-1)
    for i in range(len(axes)):
        if i < n:
            axes[i].imshow(images[i], cmap=cmap)
            axes[i].set_title(titres[i], fontsize=10)
        axes[i].axis("off")
    if titre_general:
        fig.suptitle(titre_general, fontsize=14, fontweight="bold")
    fig.tight_layout()
    return fig


def afficher_courbe_loss(historique: dict) -> plt.Figure:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(historique["losses"]); ax1.set_yscale("log")
    ax1.set_xlabel("Itération"); ax1.set_ylabel("Loss (MSE)"); ax1.set_title("Courbe de loss"); ax1.grid(alpha=0.3)
    ax2.plot(historique["psnrs"], color="green")
    ax2.set_xlabel("Itération"); ax2.set_ylabel("PSNR (dB)"); ax2.set_title("PSNR"); ax2.grid(alpha=0.3)
    fig.tight_layout()
    return fig


def rapport_visuel_complet(
    modele, enc_position, donnees: dict, historique: dict,
    device: torch.device | str = "cpu", angles_nouveaux: Optional[List[float]] = None,
    nb_points: int = 128, dossier_sortie: Optional[str] = "renders_ct",
) -> None:
    """Génère : vérité vs rendu, nouvelles vues, coupe reconstruite, courbes — et sauvegarde chaque figure en PNG."""
    geometrie: GeometrieCT = donnees["geometrie"]
    proche, loin = bornes_integration(geometrie)
    angles_train = donnees["angles"].tolist()
    projections_train = donnees["projections"].numpy()

    if dossier_sortie:
        os.makedirs(dossier_sortie, exist_ok=True)

    def _afficher_et_sauver(fig: plt.Figure, nom_fichier: str) -> None:
        fig.show()
        if dossier_sortie:
            chemin = f"{dossier_sortie}/{nom_fichier}"
            fig.savefig(chemin, dpi=120, bbox_inches="tight")
            print(f"  → sauvegardé : {chemin}")

    indices_echantillon = np.linspace(0, len(angles_train) - 1, min(4, len(angles_train))).astype(int)
    images_gt, images_rendu, titres_gt, titres_rendu = [], [], [], []
    for i in indices_echantillon:
        angle = angles_train[i]
        images_gt.append(projections_train[i]); titres_gt.append(f"Vérité — {np.degrees(angle):.0f}°")
        images_rendu.append(rendre_projection_ct(modele, enc_position, geometrie, angle, proche, loin, nb_points, device))
        titres_rendu.append(f"Rendu — {np.degrees(angle):.0f}°")

    print("Projections d'entraînement : vérité terrain vs rendu")
    _afficher_et_sauver(afficher_grille_projections(images_gt + images_rendu, titres_gt + titres_rendu), "01_verite_vs_rendu.png")

    if angles_nouveaux is None:
        angles_nouveaux = list(np.linspace(0, np.pi, 6, endpoint=False))
    images_nouvelles, titres_nouvelles = [], []
    for angle in angles_nouveaux:
        images_nouvelles.append(rendre_projection_ct(modele, enc_position, geometrie, angle, proche, loin, nb_points, device))
        titres_nouvelles.append(f"Nouvel angle — {np.degrees(angle):.0f}°")

    print("Synthèse de nouvelles vues")
    _afficher_et_sauver(afficher_grille_projections(images_nouvelles, titres_nouvelles), "02_nouvelles_vues.png")

    coupe = rendre_coupe_ct(modele, enc_position, geometrie, axe="z", niveau=0.0, device=device)
    print("Coupe reconstruite (plan z=0)")
    _afficher_et_sauver(afficher_grille_projections([coupe], ["Coupe reconstruite — z=0"], cmap="bone"), "03_coupe_reconstruite.png")

    print("Courbes d'entraînement")
    _afficher_et_sauver(afficher_courbe_loss(historique), "04_courbes_entrainement.png")


In [ ]:
%%writefile src/scene3d_ct.py
# ════════════════════════════════
# Fichier : scene3d_ct.py
# Rôle    : Génère la scène 3D tournante (rendu final) à partir du champ
#           d'atténuation appris par le NeRF-CT :
#             1. Évalue μ(x) sur une grille 3D régulière (le "volume" complet)
#             2. Extrait une surface (marching cubes, scikit-image)
#             3. Rend un tour complet (rotation 360°) en GIF
#
# Complète visuellement rapport_visuel_complet (qui reste en 2D) : c'est la
# vraie "scène 3D" demandée pour la livraison.
# ════════════════════════════════

from __future__ import annotations

import os
from typing import Optional, Tuple

import numpy as np
import torch


@torch.no_grad()
def evaluer_volume_grille(
    modele, enc_position, geometrie, resolution: int = 128,
    device: torch.device | str = "cpu", taille_lot: int = 65536,
) -> np.ndarray:
    """
    Évalue μ(x) sur une grille régulière (Z, Y, X) couvrant tout le volume —
    nécessaire pour l'extraction de mesh (marching cubes).
    """
    device = torch.device(device)
    taille_z, taille_y, taille_x = geometrie.taille_volume

    lin_z = torch.linspace(-taille_z / 2, taille_z / 2, resolution, device=device)
    lin_y = torch.linspace(-taille_y / 2, taille_y / 2, resolution, device=device)
    lin_x = torch.linspace(-taille_x / 2, taille_x / 2, resolution, device=device)
    grille_z, grille_y, grille_x = torch.meshgrid(lin_z, lin_y, lin_x, indexing="ij")

    points = torch.stack([grille_x, grille_y, grille_z], dim=-1).reshape(-1, 3)

    sorties = []
    for debut in range(0, points.shape[0], taille_lot):
        lot = points[debut : debut + taille_lot]
        position_encodee = enc_position(lot)
        sorties.append(modele(position_encodee).cpu())

    volume_attenuation = torch.cat(sorties).reshape(resolution, resolution, resolution).numpy()
    return volume_attenuation


def generer_scene_3d_tournante(
    modele, enc_position, geometrie,
    chemin_sortie: str = "renders_ct/scene_3d_tournante.gif",
    resolution: int = 96,
    percentile_seuil: float = 85.0,
    nb_frames: int = 36,
    device: torch.device | str = "cpu",
    couleur_surface: Tuple[float, float, float] = (0.82, 0.78, 0.75),
) -> str:
    """
    Génère et sauvegarde une scène 3D tournante (GIF) du volume reconstruit.

    Args:
        percentile_seuil: seuil d'isosurface — percentile de la distribution
            des valeurs d'atténuation (85 = ne garde que les 15% de points
            les plus denses, typiquement les structures les plus opaques).
            À ajuster si le mesh est vide (baisser) ou trop englobant (monter).
        nb_frames: nombre d'images de la rotation 360° (36 = tous les 10°).

    Returns:
        Chemin du GIF généré.
    """
    from skimage import measure
    import matplotlib.pyplot as plt
    from mpl_toolkits.mplot3d.art3d import Poly3DCollection
    import imageio.v2 as imageio

    print(f"Évaluation du volume sur une grille {resolution}³...")
    volume_attenuation = evaluer_volume_grille(modele, enc_position, geometrie, resolution, device)

    seuil = float(np.percentile(volume_attenuation, percentile_seuil))
    print(f"Seuil d'isosurface (percentile {percentile_seuil}) : {seuil:.4f}")

    try:
        verts, faces, _, _ = measure.marching_cubes(volume_attenuation, level=seuil)
    except (ValueError, RuntimeError) as e:
        raise RuntimeError(
            f"Marching cubes a échoué avec le seuil {seuil:.4f} (min={volume_attenuation.min():.4f}, "
            f"max={volume_attenuation.max():.4f}). Essayez un percentile_seuil différent."
        ) from e

    print(f"Mesh extrait : {len(verts)} sommets, {len(faces)} faces.")

    if chemin_sortie:
        os.makedirs(os.path.dirname(chemin_sortie) or ".", exist_ok=True)

    frames = []
    limites = (0, resolution)
    for i, azim in enumerate(np.linspace(0, 360, nb_frames, endpoint=False)):
        fig = plt.figure(figsize=(6, 6), dpi=100)
        ax = fig.add_subplot(111, projection="3d")
        mesh = Poly3DCollection(verts[faces], alpha=1.0)
        mesh.set_facecolor(couleur_surface)
        mesh.set_edgecolor((0.2, 0.2, 0.2, 0.05))
        ax.add_collection3d(mesh)
        ax.set_xlim(*limites); ax.set_ylim(*limites); ax.set_zlim(*limites)
        ax.set_box_aspect((1, 1, 1))
        ax.view_init(elev=15, azim=azim)
        ax.set_axis_off()
        fig.tight_layout(pad=0)
        fig.canvas.draw()

        image = np.asarray(fig.canvas.buffer_rgba())[..., :3]
        frames.append(image)
        plt.close(fig)

        if (i + 1) % 10 == 0:
            print(f"  frame {i + 1}/{nb_frames}")

    imageio.mimsave(chemin_sortie, frames, fps=12, loop=0)
    print(f"Scène 3D tournante sauvegardée : {chemin_sortie} ({nb_frames} frames)")
    return chemin_sortie


def generer_turntable_projections(
    modele, enc_position, geometrie, proche: float, loin: float,
    chemin_sortie: str = "renders_ct/turntable_projections.gif",
    nb_frames: int = 36, nb_points: int = 128,
    device: torch.device | str = "cpu",
) -> str:
    """
    Turntable complémentaire, plus simple et toujours fiable : rotation 360°
    des PROJECTIONS rendues (vues radiographiques), sans passer par
    l'extraction de mesh. Utile si marching_cubes échoue ou en complément.
    """
    from visualiser_ct import rendre_projection_ct
    import imageio.v2 as imageio

    if chemin_sortie:
        os.makedirs(os.path.dirname(chemin_sortie) or ".", exist_ok=True)

    frames = []
    for angle in np.linspace(0, 2 * np.pi, nb_frames, endpoint=False):
        image = rendre_projection_ct(modele, enc_position, geometrie, float(angle), proche, loin, nb_points, device)
        etendue = image.max() - image.min()
        image_normalisee = (255 * (image - image.min()) / (etendue + 1e-8)).astype(np.uint8)
        frames.append(image_normalisee)

    imageio.mimsave(chemin_sortie, frames, fps=12, loop=0)
    print(f"Turntable de projections sauvegardé : {chemin_sortie}")
    return chemin_sortie


In [ ]:
%%writefile scripts/prepare_ct_data.py
# ════════════════════════════════
# Fichier : prepare_ct_data.py
# Rôle    : Charge un volume CT (DICOM), simule des projections cone-beam
#           cohérentes avec la géométrie d'entraînement, et sauvegarde le
#           tout au format attendu par dataset_ct.py.
#
# CHANGEMENT vs Sprint 1 : n'utilise plus TIGRE (install cassée sur Colab)
# ni l'approximation parallèle-beam (--simple_projector, cause probable du
# flou observé). Utilise geometrie_ct.simuler_projections_conebeam, qui
# partage EXACTEMENT le même modèle de rayons que l'entraînement.
#
# Usage :
#   python prepare_ct_data.py --dicom_dir chemin/vers/patient \
#       --sortie data/ct_pancreas_pilote --n_angles 90 --resolution_volume 128
#
#   Mode test sans vraie donnée :
#   python prepare_ct_data.py --mock --sortie data/ct_mock --n_angles 20
# ════════════════════════════════

from __future__ import annotations

import argparse
import json
import os

import numpy as np

from geometrie_ct import GeometrieCT, simuler_projections_conebeam


def _charger_volume_dicom(dicom_dir: str):
    """
    Charge une série DICOM en volume 3D + espacement voxel RÉEL (lu depuis
    les métadonnées DICOM — PixelSpacing pour X/Y, écart réel entre
    ImagePositionPatient pour Z, plus robuste que SliceThickness qui peut
    ne pas correspondre à l'espacement inter-coupes effectif).
    """
    import pydicom

    fichiers = [f for f in os.listdir(dicom_dir) if f.lower().endswith(".dcm")]
    if not fichiers:
        raise FileNotFoundError(f"Aucun fichier .dcm trouvé dans {dicom_dir}")

    slices = [pydicom.dcmread(os.path.join(dicom_dir, f)) for f in fichiers]
    slices.sort(key=lambda s: float(s.ImagePositionPatient[2]))

    volume = np.stack([s.pixel_array.astype(np.float32) for s in slices], axis=0)

    slope = float(getattr(slices[0], "RescaleSlope", 1.0))
    intercept = float(getattr(slices[0], "RescaleIntercept", 0.0))
    volume_hu = volume * slope + intercept
    # HU → coefficient d'atténuation relatif, normalisé positif (≥0).
    volume_mu = np.clip(volume_hu + 1000.0, 0, None) / 1000.0

    pixel_spacing = [float(v) for v in slices[0].PixelSpacing]  # [espacement_ligne, espacement_colonne] mm
    positions_z = sorted(float(s.ImagePositionPatient[2]) for s in slices)
    ecarts_z = np.diff(positions_z)
    espacement_z = float(np.median(ecarts_z)) if len(ecarts_z) > 0 else float(
        getattr(slices[0], "SliceThickness", 1.0)
    )

    taille_voxel_mm = (espacement_z, pixel_spacing[0], pixel_spacing[1])  # (Z, Y, X)
    return volume_mu.astype(np.float32), taille_voxel_mm


def _redimensionner_volume(volume: np.ndarray, taille_voxel_mm, resolution_cible: int):
    """
    Réduit le volume à ``resolution_cible`` sur chaque axe (les CT bruts sont
    souvent 512×512×N, trop lourds pour un entraînement NeRF rapide sur un
    seul GPU Colab). Ajuste la taille de voxel en conséquence pour garder
    les dimensions PHYSIQUES du volume identiques.
    """
    from scipy.ndimage import zoom

    facteurs = [resolution_cible / s for s in volume.shape]
    volume_reduit = zoom(volume, facteurs, order=1)
    taille_voxel_ajustee = tuple(t / f for t, f in zip(taille_voxel_mm, facteurs))
    return volume_reduit.astype(np.float32), taille_voxel_ajustee


def _volume_mock(taille=(96, 96, 96)) -> np.ndarray:
    z, y, x = np.mgrid[0 : taille[0], 0 : taille[1], 0 : taille[2]]
    centre = np.array(taille) / 2
    rayon = min(taille) * 0.3
    distance = np.sqrt((z - centre[0]) ** 2 + (y - centre[1]) ** 2 + (x - centre[2]) ** 2)
    return ((distance < rayon).astype(np.float32) * 0.8)


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--dicom_dir", type=str, default=None)
    parser.add_argument("--mock", action="store_true")
    parser.add_argument("--sortie", type=str, required=True)
    parser.add_argument("--n_angles", type=int, default=90)
    parser.add_argument("--resolution_volume", type=int, default=128, help="Volume réduit à N³ voxels avant simulation")
    parser.add_argument("--resolution_detecteur", type=int, default=160)
    parser.add_argument("--nb_points_integration", type=int, default=256)
    parser.add_argument("--split_val", type=float, default=0.1)
    parser.add_argument("--device", type=str, default="cuda", help="cuda si disponible pour accélérer la simulation")
    args = parser.parse_args()

    import torch
    device = args.device if (args.device == "cpu" or torch.cuda.is_available()) else "cpu"

    os.makedirs(args.sortie, exist_ok=True)
    angles = np.linspace(0, np.pi, args.n_angles, endpoint=False).astype(np.float32)

    if args.mock:
        volume = _volume_mock((args.resolution_volume,) * 3)
        taille_voxel_mm = (2.0, 2.0, 2.0)
    else:
        if not args.dicom_dir:
            raise ValueError("--dicom_dir requis hors mode --mock")
        volume, taille_voxel_mm = _charger_volume_dicom(args.dicom_dir)
        volume, taille_voxel_mm = _redimensionner_volume(volume, taille_voxel_mm, args.resolution_volume)

    taille_volume_mm = tuple(float(t * v) for t, v in zip(volume.shape, taille_voxel_mm))
    demi_diag = 0.5 * float(np.linalg.norm(taille_volume_mm))

    geometrie = GeometrieCT(
        DSO=max(2.5 * demi_diag, 300.0),
        DSD=max(4.0 * demi_diag, 500.0),
        n_detecteur=(args.resolution_detecteur, args.resolution_detecteur),
        taille_pixel_detecteur=(demi_diag * 2.2 / args.resolution_detecteur,) * 2,
        taille_volume=taille_volume_mm,
    )

    print(f"Volume : {volume.shape}, taille physique {taille_volume_mm} mm")
    print(f"Simulation de {len(angles)} projections cone-beam ({device})...")
    projections = simuler_projections_conebeam(
        volume, geometrie, angles, device=device, nb_points_integration=args.nb_points_integration
    )

    n_val = max(1, int(len(angles) * args.split_val))
    indices = np.random.permutation(len(angles))
    idx_val, idx_train = indices[:n_val], indices[n_val:]

    for nom_split, idx in [("train", idx_train), ("val", idx_val)]:
        sous_dossier = os.path.join(args.sortie, nom_split)
        os.makedirs(sous_dossier, exist_ok=True)
        np.save(os.path.join(sous_dossier, "angles.npy"), angles[idx])
        np.save(os.path.join(sous_dossier, "projections.npy"), projections[idx])

    geometrie_json = {
        "DSO": geometrie.DSO,
        "DSD": geometrie.DSD,
        "n_detecteur": list(geometrie.n_detecteur),
        "taille_pixel_detecteur": list(geometrie.taille_pixel_detecteur),
        "taille_volume": list(geometrie.taille_volume),
    }
    with open(os.path.join(args.sortie, "geometrie.json"), "w") as f:
        json.dump(geometrie_json, f, indent=2)

    # Volume de référence sauvegardé — utile pour comparer visuellement la
    # reconstruction NeRF à la vérité terrain (voir scene3d_ct.py).
    np.save(os.path.join(args.sortie, "volume_verite.npy"), volume)

    print(f"Dataset CT préparé dans {args.sortie} : {len(idx_train)} projections train, {len(idx_val)} val.")


if __name__ == "__main__":
    main()


## 6. Test rapide — mode `--mock` (sans vraie donnée)

Valide que tout le pipeline tourne avant de passer aux vraies données.
Quelques secondes seulement.

In [ ]:
!python scripts/prepare_ct_data.py --mock --sortie data/ct_mock \
    --n_angles 20 --resolution_volume 48 --resolution_detecteur 64 --device cpu

In [ ]:
import importlib, sys
for m in ['geometrie_ct','nerf_model_ct','dataset_ct','train_ct','visualiser_ct','scene3d_ct']:
    sys.modules.pop(m, None)

from train_ct import entrainer_nerf_ct
modele, historique = entrainer_nerf_ct('data/ct_mock', nb_iterations=100, taille_batch=512, nb_points=32, sauvegarder_tous=0)
print("Test mock OK — loss finale:", historique['losses'][-1])

## 7. Vos données réelles — Pancreas-CT

Si ce n'est pas déjà fait : montez votre Drive et téléchargez le dataset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Si le dataset n'est pas encore sur votre Drive (sinon, passez cette cellule) :
!pip install kaggle -q
# uploadez votre kaggle.json (Compte Kaggle → Create API Token) puis :
# !kaggle datasets download -d eliasmarcon/pancreas -p /content/drive/MyDrive/pancreas_ct
# !unzip -q /content/drive/MyDrive/pancreas_ct/pancreas.zip -d /content/drive/MyDrive/pancreas_ct

In [ ]:
# Repérer le dossier d'un patient contenant directement les fichiers .dcm :
!find /content/drive/MyDrive/pancreas_ct -iname "*.dcm" | head -3

## 8. Préparation des données — simulateur cone-beam

Remplacez `DICOM_DIR` par le chemin trouvé ci-dessus (dossier parent des
`.dcm`). `resolution_volume` contrôle la taille du volume utilisé — 128 est
un bon compromis qualité/temps sur un T4 ; montez à 160-192 si vous avez du
temps et de la mémoire GPU en réserve.

In [ ]:
DICOM_DIR = "/content/drive/MyDrive/pancreas_ct/Pancreas_CT/PANCREAS_0007"  # ⚠ à adapter

!python scripts/prepare_ct_data.py --dicom_dir "{DICOM_DIR}" \
    --sortie data/ct_pancreas_final --n_angles 90 \
    --resolution_volume 128 --resolution_detecteur 160 \
    --nb_points_integration 256 --device cuda

## 9. Entraînement — qualité finale

Valeurs par défaut pensées pour un résultat de qualité sur GPU T4 (pas
juste "code tournable"). Ordre de grandeur : plusieurs dizaines de minutes
sur T4 selon la charge Colab — les checkpoints réguliers permettent de
reprendre si la session est interrompue (`checkpoint_reprise=...`).

Si le temps presse, réduisez `nb_iterations` à 4000-5000 : la loss aura
déjà bien convergé (cf. votre test Sprint 1 où la stabilisation démarrait
vers 1000-1500 itérations).

In [ ]:
for m in ['geometrie_ct','nerf_model_ct','dataset_ct','train_ct']:
    sys.modules.pop(m, None)
from train_ct import entrainer_nerf_ct

modele, historique = entrainer_nerf_ct(
    'data/ct_pancreas_final',
    nb_iterations=8000,
    taille_batch=2048,
    nb_points=128,
    sauvegarder_tous=1000,
    dossier_checkpoints='checkpoints_ct_final/',
)
print("Loss finale :", historique['losses'][-1])
print("PSNR finale :", historique['psnrs'][-1])

## 10. Retours visuels 2D

Vérité terrain vs rendu, nouvelles vues (angles jamais vus à
l'entraînement), coupe 2D reconstruite, courbes de convergence. Sauvegardés
dans `renders_ct_final/`.

In [ ]:
from dataset_ct import charger_dataset_ct
from visualiser_ct import rapport_visuel_complet

donnees = charger_dataset_ct('data/ct_pancreas_final', split='train')
rapport_visuel_complet(
    modele, historique['enc_position'], donnees, historique,
    device='cuda', nb_points=128, dossier_sortie='renders_ct_final',
)

## 11. Scène 3D tournante — le livrable final

Extrait une surface 3D du champ appris (marching cubes) et génère une
rotation 360° en GIF.

`percentile_seuil` contrôle l'isosurface extraite : augmentez-le (ex. 90-95)
si le mesh englobe trop de bruit, baissez-le (ex. 70-80) si rien
n'apparaît. Ajustez et relancez cette cellule seule — pas besoin de
ré-entraîner.

In [ ]:
from scene3d_ct import generer_scene_3d_tournante

chemin_scene = generer_scene_3d_tournante(
    modele, historique['enc_position'], donnees['geometrie'],
    chemin_sortie='renders_ct_final/scene_3d_tournante.gif',
    resolution=128, percentile_seuil=85, nb_frames=48,
    device='cuda',
)

In [ ]:
# Afficher le GIF directement dans le notebook
from IPython.display import Image, display
display(Image(filename=chemin_scene))

### Turntable de secours (projections 2D tournantes)

Si le mesh 3D n'est pas satisfaisant (bruit, seuil difficile à régler),
cette version plus simple tourne les projections rendues — toujours fiable,
moins spectaculaire.

In [ ]:
from geometrie_ct import bornes_integration
from scene3d_ct import generer_turntable_projections

proche, loin = bornes_integration(donnees['geometrie'])
generer_turntable_projections(
    modele, historique['enc_position'], donnees['geometrie'], proche, loin,
    chemin_sortie='renders_ct_final/turntable_projections.gif',
    nb_frames=36, nb_points=128, device='cuda',
)

## 12. Télécharger les livrables

Récupère tout le dossier de rendus (GIFs + PNGs) en une archive.

In [ ]:
import shutil
shutil.make_archive('livrables_nerf_ct', 'zip', 'renders_ct_final')
from google.colab import files
files.download('livrables_nerf_ct.zip')